In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("ps09.ipynb")

# PS9 — Evolutionary Failure, Communities, Populations and Classifiers
### BioE 147/247 · Fall 2026

**Out:** Thursday, November 19 · **Due:** Thursday, December 3, 11:59 pm
**Covers:** Sessions 23, 24, 25 and 26 · **50 points** · **Question 11** is required for **BioE 247** (58 total) and is extra credit for **BioE 147** (up to +8)

---

One question, asked four times at four scales: **what does this design do to a
population you are not watching?**

Questions 1–3 are one strain losing a kill switch, and containment turns out
not to be a number — it is a duration. Questions 4–6 are cells signalling to
each other, where the receiver is a density meter and the *geometry* decides
what the circuit can possibly know. Questions 7–8 are a parasite engineered so
that it spreads, where $R_0 > 1$ is the design target rather than the failure.
Questions 9–10 are a classifier deciding which cells to kill, and the deciding
turns out to be one comparison made outside the circuit entirely.

**This set posts before two of its four sessions have happened.** That is
deliberate — you get the whole thing early — but it means you should start at
the top and not worry about the bottom:

| questions | you can start | after |
|---|---|---|
| **1–3** | immediately | session 23 (Nov 12) |
| **4–6** | immediately | session 24 (Nov 17) |
| **7–8** | after Nov 19 | session 25 |
| **9–11** | after Nov 24 | session 26 |

Nine days remain after session 26. If you work Q1–Q6 in the first week the
rest is comfortable; if you wait, it is not.

**A note on precision.** Report numeric answers to **at least four significant
figures**. Every graded value on this set is checked to a relative tolerance of
$10^{-3}$, so four figures is a full order of magnitude of slack — but
rounding to two or three will fail, and so will reporting a number you
remembered instead of one you computed. Several questions also check that your
answers are consistent *with each other*, which a remembered number cannot be.

**Collaboration is encouraged.** Discuss, argue, work at a whiteboard
together, then write your own solution and your own code. Record who you
worked with below.

**If you used an LLM**, say so briefly and say what for. The conditions are
that you can explain anything you submit and that the code you submit runs.

**A note on the visible tests.** They check *properties* — limits,
invariances, signs, orderings. A green visible check means "not obviously
broken", not "right".

Fill in the two cells below before you start.

In [ ]:
COLLABORATORS = ""   # e.g. "worked with J. Chen on Q5"
AI_USE = ""          # e.g. "used an LLM to check my algebra in Q8"

## Setup

In [ ]:
# ---------------------------------------------------------------------------
# SETUP — run this cell first, every time.
#
# DataHub / local : finds the repository root and puts it on the import path.
# Google Colab    : clones the repository first, because Colab opens this
#                   notebook on its own, without the posb package beside it.
# ---------------------------------------------------------------------------
import os
import sys

if "google.colab" in sys.modules:
    if not os.path.exists("posb2026"):
        !git clone -q https://github.com/ArkinLaboratory/posb2026.git
    sys.path.insert(0, os.path.abspath("posb2026"))
else:
    _d = os.getcwd()
    while _d != os.path.dirname(_d) and not os.path.isdir(os.path.join(_d, "posb")):
        _d = os.path.dirname(_d)
    sys.path.insert(0, _d)

import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

import posb
from posb import Reaction, Model

posb.check_environment()


In [ ]:
from posb.evolution import (time_to_failure, broken_fraction,
                            fitness_cost_from_competition,
                            fitness_cost_from_growth,
                            containment_requirement, escapers_per_dose,
                            retention_time)
from posb.communities import (sender_receiver, basu_params, band_peak,
                              band_edges, band_detect_response,
                              penetration_depth, diffusion_time)
from posb.populations import (diploid_packaging, infectious_fraction,
                              basic_R0, therapeutic_R0, P_min)
import posb.classifiers as cf
from scipy.optimize import brentq

---
## Question 1 — How long does the switch last?  *(6 points)*

A CRISPR kill switch sits on a plasmid in a probiotic chassis. Two numbers
describe how it is lost.

- **$s = 0.02$ per generation** is the growth advantage a cell gets from losing
  the switch: a cell with a working switch grows at $(1-s)$ times the rate of a
  cell whose switch is broken.
- **$\mu = 1.0 \times 10^{-4}$ per cell per generation** is the probability that
  a working switch breaks in one generation. It is a **lumped** rate: this
  construct is plasmid-borne, and a daughter that fails to inherit the plasmid
  is counted here alongside a daughter whose cassette mutated. You are told
  this; you are not asked to derive it.

$q$ is the fraction of the population whose switch no longer works, $q(0) = 0$,
and **time is in generations throughout, never hours**. The model is

$$\dot q = (1-q)(sq + \mu), \qquad
t(q) = \frac{\ln\!\big[(sq+\mu)\,/\,\mu(1-q)\big]}{s+\mu}$$

**Q1.**

**(a)** Set `T50` to the number of generations until half the population has
lost the switch.

**(b)** Set `Q_100` to the broken fraction after 100 generations of permissive
growth.

**(c)** You have budget for exactly one of two redesigns, and they are the two
the hour priced:

- **halve the burden** — $s \to s/2$, with $\mu$ unchanged;
- **harden the DNA a hundredfold** — $\mu \to \mu/100$, with $s$ unchanged.

Set `GAIN_HALVE_S` and `GAIN_MU_OVER_100` to the extra generations of half-life
each move buys, **relative to your answer in (a)**, and set `BETTER` to the
string `"s"` or `"mu"` for the one that buys more **on this construct**.

**(d)** Now price the losing move. Holding $s$ at $0.02$, set `FOLD` to the
factor by which $\mu$ would have to be **divided** for the half-life to equal
what halving the burden achieves — that is, solve
$t_{50}(s,\ \mu/F) = t_{50}(s/2,\ \mu)$ for $F$. A root-finder is the intended
tool.

One line in the margin for yourself, not graded: `FOLD` is the exchange rate
between the two engineering programmes **at this** $s/\mu$, and it is not the
exchange rate at the one in lecture.

In [ ]:
s, mu = 0.02, 1.0e-4

...

T50, Q_100, GAIN_HALVE_S, GAIN_MU_OVER_100, BETTER, FOLD

In [ ]:
grader.check("q1")

---
## Question 2 — What does the switch actually cost?  *(5 points)*

Question 1 handed you $s = 0.02$. Nobody hands you $s$. Here are two
measurements of the same strain, made in the same week by the same person.

Throughout this question $s$ is defined by

$$\frac{\mathrm{d}}{\mathrm{d}g}\ln\frac{n_{\text{circuit}}}{n_{\text{control}}} = -s$$

with $g$ in generations of the control strain — so $s$ is **minus the slope of
the log ratio**, and a strain with no cost gives a flat line.

**Measurement A — a head-to-head competition.** The circuit strain and an
empty-vector control were mixed 1:1 and passaged together. At each sampling the
two were counted by plating on selective and non-selective medium, and the table
gives $n_{\text{circuit}}/n_{\text{control}}$:

| generations | 0 | 10 | 20 | 30 | 40 | 50 |
|---|---|---|---|---|---|---|
| ratio | 1.000 | 0.842 | 0.690 | 0.588 | 0.481 | 0.409 |

**Measurement B — two growth curves, run separately.** Each strain was grown in
its own flask in the same medium at 37 °C, and density was read hourly.
Densities are in **units of $10^8$ cells / mL**:

| time (h) | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| control | 0.0100 | 0.0332 | 0.110 | 0.366 | 1.22 | 4.03 | 13.4 |
| circuit | 0.0100 | 0.0320 | 0.103 | 0.329 | 1.05 | 3.37 | 10.8 |

**Q2a.** Set `S_COMPETITION` to the selection coefficient estimated
from Measurement A, and `S_GROWTH` to the one estimated from Measurement B.

**Both are least-squares fits, not two-point slopes.** For A, take the
least-squares slope of $\ln(\text{ratio})$ against generations, over all six
points, and report minus it. For B, fit each strain's exponential growth rate by
least squares on $\ln(\text{density})$ against time, over all seven points, and
report $1 - (\text{circuit rate})/(\text{control rate})$. Reading a slope off
the two endpoints gives a different number here, by about one per cent, which is
more than the grader allows.

In [ ]:
GENERATIONS = [0, 10, 20, 30, 40, 50]
RATIO = [1.000, 0.842, 0.690, 0.588, 0.481, 0.409]

HOURS = [0, 1, 2, 3, 4, 5, 6]
CONTROL = [0.0100, 0.0332, 0.110, 0.366, 1.22, 4.03, 13.4]
CIRCUIT = [0.0100, 0.0320, 0.103, 0.329, 1.05, 3.37, 10.8]

...

S_COMPETITION, S_GROWTH

In [ ]:
grader.check("q2a")

<!-- BEGIN QUESTION -->

**Q2b.** The two numbers disagree substantially — far more than either
measurement's own scatter would explain. They are not two noisy draws from one
quantity; one design of experiment controls something the other does not.

**(i)** Name what the competition assay holds fixed between the two strains
that two separate flasks do not, and say why that matters when the quantity you
are after is a few per cent of a growth rate. One or two sentences.

**(ii)** You must report one number to a colleague who will put it into
Question 1's model. Say which you report and why — and say what in Measurement B
you would have to show before the other number could be believed. Do not argue
from which answer you prefer.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 3 — Containment is a duration  *(4 points)*

The same chassis is now a therapeutic, and the capsule is not a mouse gavage.
One oral dose is **$5 \times 10^{10}$ CFU**. The containment standard is the one
the hour used: **fewer than one escaper per dose.**

The plating assay says this strain's fraction viable on induction is
**$10^{-8.6}$** — the figure Rottinghaus *et al.* report, and the best number in
that paper.

After the redesign that removed the internal repeat from the promoter, escape
requires **all three** gRNA cassettes to be lost, each independently, at
**$\mu_1 = 2.0 \times 10^{-4}$ per cassette per generation**. With $k$ cassettes
each still intact with probability $e^{-\mu_1 t}$,

$$q(t) = \left(1 - e^{-\mu_1 t}\right)^{k}$$

and $t$ is in generations of permissive growth — no inducer, the population
doubling normally. Note $\mu_1$ is **per cassette** and is not Question 1's
whole-circuit $\mu$.

**Q3.**

**(a)** Set `REQUIRED_FV` to the fraction viable this dose requires.

**(b)** Set `ESCAPERS` to the expected number of survivors in one dose at the
measured fraction viable of $10^{-8.6}$.

**(c)** Set `RETENTION` to the number of generations of permissive growth after
which the standing escape fraction first exceeds `REQUIRED_FV`, starting from
none broken.

**(d)** Set `DOSE_MAX` to the largest dose, in CFU, for which a strain sitting
at $10^{-8.6}$ delivers fewer than one escaper.

Then, in one sentence in the markdown cell below: the assay result and your
answer to (b) are both true of the same strain. Say what each of the two numbers
is a statement **about**. *(Not separately graded; it is the sentence the set's
closing paragraph picks up.)*

In [ ]:
DOSE = 5.0e10
FRACTION_VIABLE = 10 ** -8.6
MU1 = 2.0e-4

...

REQUIRED_FV, ESCAPERS, RETENTION, DOSE_MAX

In [ ]:
grader.check("q3")

---
## Question 4 — A receiver is a density meter  *(5 points)*

One strain makes a signal and another reads it. That is the whole of a
sender/receiver link, and it is two equations.

**The sender.** $A$ is the signal concentration in **nM**, shared by one
well-mixed volume. $S$ is the sender density, in units of $10^{8}$ cells / mL.
The sender makes signal in proportion to how many senders there are, the medium
destroys it first-order, and a reactor dilutes it out as well:

$$\frac{\mathrm{d}A}{\mathrm{d}t} = k_I S - \gamma_A A - D A$$

- $k_I = 0.030$ **nM min⁻¹ per $10^{8}$ cells / mL** — the synthase expression,
  which is a promoter choice. It is yours; neither paper prints one.
- $\gamma_A = 0.025$ **min⁻¹** — the signal's first-order decay in **this**
  medium. Question 6 runs a different medium and a different signal, and its
  decay rate is not this one; the constant here is seeded as `GAMMA_MEDIUM` so
  the two cannot collide in your notebook.
- $D$ is the **vessel's dilution rate**, in min⁻¹. On a plate $D = 0$. In this
  question's chemostat $D = 0.010$ min⁻¹, which is 0.6 h⁻¹.

On this set a bare $D$ is a dilution rate and never a diffusion coefficient,
exactly as in the hour.

**The receiver.** Its output is a Hill function of the signal it sees:

$$\mathrm{output} = \frac{\alpha\,A^{n}}{\theta_A^{\,n} + A^{n}}$$

with $\alpha = 1$, so the output is a **fraction of its own maximum**, between 0
and 1; $\theta_A = 5.0$ **nM** is the signal concentration at half output; and
$n = 2$ is the sharpness. $\theta_A$ is not a saturation level.

Set $\mathrm{d}A/\mathrm{d}t = 0$ and the signal the receiver sees is one line:

$$A^* = \frac{k_I S}{\gamma_A + D}$$

so the receiver's output is a **report on sender density**, read through the
Hill function, with the vessel's dilution competing directly against the
molecule's own decay.

You are running $S = 4.0$ (that is $4.0 \times 10^{8}$ cells / mL) of senders.

**Q4.**

**(a)** Set `A_PLATE` to the steady-state signal in **nM** on a plate, and
`A_REACTOR` to the steady-state signal in **nM** in the chemostat.

**(b)** Set `OUT_PLATE` and `OUT_REACTOR` to the receiver's output in each, as a
fraction of maximum.

**(c)** You want the receiver sitting at **half** its maximum output in the
chemostat, and you may not change the medium, the promoter or the receiver. Set
`S_NEEDED` to the sender density, in units of $10^{8}$ cells / mL, that puts it
there.

In [ ]:
K_I, GAMMA_MEDIUM, D_REACTOR = 0.030, 0.025, 0.010
THETA_A, N_HILL = 5.0, 2.0
S_SENDER = 4.0

...

A_PLATE, A_REACTOR, OUT_PLATE, OUT_REACTOR, S_NEEDED

In [ ]:
grader.check("q4")

<!-- BEGIN QUESTION -->

**Q4d.** The receivers are now given a second gene: **AiiA**, a
lactonase that destroys the signal. Receiver density is $R$, in the same units
as $S$, and the lactonase removes signal at $\gamma_E R A$ with $\gamma_E$ a
rate per unit receiver density.

**(i)** Write the sender equation again with that term in it, and give the new
$A^*$. One line each.

**(ii)** Your new $A^*$ has three removal terms in its denominator and they
enter it **identically**. Say what that means for a colleague who measures only
$A^*$ and wants to know whether their signal is short-lived because of the
medium or because of the receivers — and name the one measurement that separates
them. Two or three sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 5 — Where the window starts and stops  *(4 points)*

Basu's band detector lights in a window of AHL because two repression routes act
at concentrations far apart: CI shuts the reporter down at high signal, the
mutant LacI shuts it down at low signal, and in between neither is effective.
**You are given that separation and you are not asked to argue for it.** What
this question asks for is the window's two edges, for a receiver that is not the
one in the paper.

In the **liquid-phase limit** — the paper's own, obtained by dropping its
equation (5) — AHL is an imposed dose rather than a field, and equations (1)–(4)
have a steady state that composes in one direction with no solving:

$$R = \frac{\rho_R\,[\mathrm{LuxR}]^2 A^2}{\gamma_R}
\qquad
f(R) = \frac{R}{\theta_R + R}$$

$$C = \frac{\alpha_C}{\gamma_C}f(R)
\qquad
L = \frac{1}{\gamma_L}\left[\frac{\alpha_{L1}}{1 + (C/\beta_C)^2}
+ \alpha_{L2}f(R)\right]
\qquad
G(A) = \frac{\alpha_G}{\gamma_G\left[1 + (L/\beta_L)^2\right]}$$

$A$ is AHL, $G$ is the GFP output, and the same activation $f(R)$ drives both CI
and the LacI route. $L$ has a falling term and a rising term, so $L$ has a
minimum and $G$ has a maximum: that is the band.

**The parameters.** Basu's printed BD2 set, with **one change**: this receiver
carries a weaker CI operator, so $\beta_C = 0.020\ \mu$M instead of the printed
0.008 µM. Everything else is as printed. $\beta_L$ and $\beta_C$ keep the
paper's own names and are in **µM**.

| parameter | value | | parameter | value |
|---|---|---|---|---|
| $\alpha_G$ | 2.0 µM/min | | $\gamma_G$ | 0.0692 min⁻¹ |
| $\alpha_{L1}$ | 1.0 µM/min | | $\gamma_L$ | 0.0231 min⁻¹ |
| $\alpha_{L2}$ | 1.0 µM/min | | $\gamma_C$ | 0.0692 min⁻¹ |
| $\alpha_C$ | 1.0 µM/min | | $\gamma_R$ | 0.0231 min⁻¹ |
| $\beta_L$ | 0.8 µM | | $\theta_R$ | 0.01 µM |
| $\beta_C$ | **0.020 µM** | | $\rho_R$ | 0.5 µM⁻³ min⁻¹ |
| $[\mathrm{LuxR}]$ | 0.5 µM, fixed | | | |

The exponents are the paper's: the two repression terms are squared, and the
LuxR–AHL complex drives its promoters **without** cooperativity, which is why
$f(R)$ has no exponent on it.

An edge needs a definition, and you take the paper's: an edge is where the
response crosses **half of its own peak**.

**Units, because this item mixes two.** The parameter table is in µM. Report
every **AHL** answer in **nM** (1 µM = 1000 nM). Report the GFP peak in **µM**,
the table's own unit.

**Q5.**

**(a)** Set `A_PEAK` to the AHL at the band's maximum, in nM, and `G_PEAK` to
the GFP there, in µM.

**(b)** Set `A_LO` and `A_HI` to the low and high edges of the band, in nM.

**(c)** Set `WIDTH` to the ratio `A_HI / A_LO` — how many fold in AHL the strain
is lit over.

Locate the peak before you root-find. The response is **not monotone**, and a
coarse grid straddles its maximum and reports the band as still rising.

One line in the margin for yourself, not graded: `WIDTH` is a **dose** window.
On a plate the dose is set by position, so the lit ring's thickness is that ratio
read through the signal's own decay length — which is the next question.

In [ ]:
...

A_PEAK, G_PEAK, A_LO, A_HI, WIDTH

In [ ]:
grader.check("q5")

---
## Question 6 — How far the signal gets, and when  *(4 points)*

Session 2 gave you one estimate and this hour gave you a second, and they answer
**different questions**:

$$t \approx \frac{L^2}{2D_A}
\qquad\text{how long a signal takes to get there}$$

$$\ell_A = \sqrt{\frac{D_A}{\gamma_A}}
\qquad\text{how far it gets before it decays}$$

The convention in the first is $2D_A$, not $6D_A$, as it has been since
session 2. $D_A$ is the **signal's diffusion coefficient**; on this set a bare
$D$ is a dilution rate.

With a source at $x = 0$, diffusion in and first-order decay out, the
steady-state profile is the exponential that forced $\ell_A$:

$$A(x) = A_0\,e^{-x/\ell_A}$$

**Your own numbers, and they are the point of the question.** No coefficient in
this session's folder can be put on this system: Basu print a parameter that is
not a diffusion coefficient, and Karig's two are a different molecule measured
through a different lawn. So you measured your own, in your own agarose, and
these are the values you are working from. They are this problem set's, not a
paper's.

| quantity | value |
|---|---|
| $D_A$, your signal in your agarose | $240\ \mu$m²/s, **which is 0.0144 mm²/min** |
| $\gamma_A$, its decay in that medium | $0.055$ min⁻¹ |
| $A_0$, the signal at the edge of the sender disk | $400$ nM |
| $A_{\mathrm{thr}}$, the signal your receivers need to switch on | $1.5$ nM |

**This is not Question 4's medium.** Its $\gamma_A$ was 0.025 min⁻¹; here it is
**0.055**, and the constant is seeded as `GAMMA_SIGNAL`. If you reach for
`GAMMA_MEDIUM` you will get a penetration depth of 0.76 mm instead of 0.51, and
the visible check cannot see the difference because the error cancels in the
ratio it tests.

**Work in millimetres and minutes.** $D_A$ is printed in both units above for
exactly that reason: 0.0144 mm²/min with $\gamma_A$ in min⁻¹ returns a length in
mm.

**Q6.**

**(a)** Set `ELL_A` to the signal's penetration depth, in **mm**.

**(b)** Set `X_LIT` to the distance from the sender disk, in **mm**, at which
the steady-state signal has fallen to the receivers' threshold — the outer edge
of the lit region.

**(c)** Set `T_ARRIVE` to the time, in **hours**, that session 2's estimate gives
for the signal to travel that distance. Say nothing about whether the signal is
still above threshold when it gets there; that is what (b) is for.

**(d)** You want the lit edge at **6.0 mm** instead. $D_A$ is not a dial — you
pick a signal and inherit it — and $A_0$ and $A_{\mathrm{thr}}$ are the senders
and the receivers, which are built. So the one knob left is the medium. Set
`GAMMA_TARGET` to the decay rate, in min⁻¹, that puts the lit edge at 6.0 mm,
with everything else unchanged.

Then, in one sentence in the markdown cell below: say what factor you had to
change $\gamma_A$ by, and what the square root in $\ell_A$ costs a designer who
wants twice the reach. *(Not separately graded; it is the sentence the set's
session-24 paragraph picks up.)*

In [ ]:
D_A = 0.0144        # mm^2/min
GAMMA_SIGNAL = 0.055  # min^-1
A_0, A_THR = 400.0, 1.5   # nM
X_TARGET = 6.0      # mm

...

ELL_A, X_LIT, T_ARRIVE, GAMMA_TARGET

In [ ]:
grader.check("q6")

---
## Question 7 — What fraction of a prep can infect anything?  *(5 points)*

A conditionally replicating HIV-1 vector (crHIV-1) carries an antiviral payload
and borrows every protein it needs from wild-type HIV-1 in the same cell. In a
dually infected cell you measure the cytoplasmic transcript pool and find
**6 vector transcripts for every 1 HIV-1 transcript** — that is,
**crHIV-1 : HIV-1 = 6 : 1**, so $P = 6$.

**$P$ is an abundance ratio, not a probability.** The probability built from it
is

$$q = \frac{P}{1+P}, \qquad 1 - q = \frac{1}{1+P}$$

the chance that one strand drawn from that pool is vector. Retroviruses are
diploid: each capsid takes **two** strands, and the model here is that the two
are independent draws from the same pool at equal per-molecule probability. So a
released particle is one of three things — two HIV-1 strands, one of each, or
two vector strands — with probabilities $(1-q)^2$, $2q(1-q)$ and $q^2$.

**And one rounding, which you are told rather than asked to justify.** An *et
al.* (1999) found that heterozygous lentiviral virions enter cells and then meet
a partial preintegration block; the model rounds that partial block to total, so
**heterozygotes are counted as non-infectious**. The infectious output is
therefore the two homozygous classes only, in the ratio **1 HIV-1 to $P^2$
vector**.

**Q7.**

**(a)** Set `P_HIV_HOM` to the probability that a released particle carries two
HIV-1 strands.

**(b)** Set `P_VEC_HOM` to the probability that a released particle carries two
vector strands.

**(c)** Set `INFECTIOUS` to the fraction of everything the cell releases that
can infect anything at all, under the rounding above.

**(d)** Your titer assay counts infectious units, not particles. Set
`VEC_SHARE_INF` to the fraction of the **infectious** output that is vector.

Then, in one sentence in the markdown cell below: your answers to (b) and (d)
are both true of the same prep and they are not the same number. Say what each
one is a statement **about**. *(Not separately graded; it is the sentence the
set's closing paragraph picks up.)*

In [ ]:
P_RATIO = 6.0   # crHIV-1 : HIV-1 transcripts in a dually infected cell

...

P_HIV_HOM, P_VEC_HOM, INFECTIOUS, VEC_SHARE_INF

In [ ]:
grader.check("q7")

---
## Question 8 — The floor moves when the burst size does  *(5 points)*

The same vector is now in a patient. Two knobs describe it.

- **$P$ is the packaging advantage**, the cytoplasmic
  **crHIV-1 : HIV-1 = $P : 1$** transcript ratio in a dually infected cell.
  Your construct measures **$P = 3.6$**.
- **$D$ is the antiviral effect, and it is the fraction of HIV-1 production that
  REMAINS.** So $D = 1$ is no inhibition at all and $D = 0$ is total inhibition.
  Your cargo leaves **$D = 0.25$** — it removes three quarters of viral output.
  Read $D$ this way throughout; the sense is the opposite of what the phrase
  "antiviral effect" suggests.

Here is the model, and Table 1 of the paper **as printed**. Note the units:
every concentration here is **per microlitre except $V_0$, which is printed per
millilitre**.

| symbol | meaning | value | units |
|---|---|---|---|
| $\lambda$ | birth rate of uninfected CD4⁺ T cells | 31 | cells µl⁻¹ day⁻¹ |
| $d$ | death rate of uninfected CD4⁺ T cells | 0.02 | day⁻¹ |
| $k$ | infection rate per virion | $1.875\times10^{-4}$ | µl virion⁻¹ |
| $n$ | burst size, virions per infected cell | 200 | — |
| $c$ | clearance rate of free virions | 30 | day⁻¹ |
| $T_0$ | uninfected CD4⁺ count | 800 | cells/µl |
| $V_0$ | HIV-1 set point before therapy | $10^{5}$ | **virions/ml** |

**Your patient is not Table 1's patient.** Everything above is as the paper
prints it, but this patient presents at a set point of
$V_0 = 8 \times 10^{4}$ **virions/ml**, not $10^{5}$. Use $8 \times 10^{4}$
throughout.

$\lambda$ and $k$ were not measured. They were back-solved so that the model
passes through $T_0$ and $V_0$: $k = c/(nT_0)$ and $\lambda = dT_0 + cV_0/n$.
**So $\lambda$ moves with the set point and Table 1's 31 is not your patient's
value — derive it.** ($k$ does not depend on $V_0$, so Table 1's
$1.875\times10^{-4}$ still holds.) The back-solve gives **two** expressions for
the same reproductive ratio,

$$R_0 = \frac{\lambda k n}{c\,d} \qquad\text{and}\qquad
R_0 = 1 + \frac{c V_0}{n\,d\,T_0}$$

which agree at Table 1's values and **stop agreeing the moment you perturb $n$**,
because only one of them can hold $T_0$ and $V_0$ fixed while $n$ moves.

The therapy's own reproductive ratio counts **new crHIV-1 infections from one
$I_T$ cell** — a cell that already carries the vector and is waiting for HIV-1 to
arrive — with the infection counted at the moment a cell becomes an $I_T$ cell.
On that convention

$$R_0^T = P^2 D\left(1 - \frac{1}{R_0}\right),
\qquad\text{so persistence needs}\qquad
P^2 D > \frac{1}{1 - 1/R_0} = 1 + \frac{n\,d\,T_0}{c\,V_0}$$

*(If you build the next-generation matrix the textbook way, counting virion
production as a new infection too, you get the square root of this $R_0^T$ and
you are right — the threshold at 1 is identical. Report the value on the
convention above, the one printed here.)*

**Q8.**

**(a)** Set `R0_PATIENT` to this patient's $R_0$, computed from $c$, $V_0$, $n$,
$d$ and $T_0$.

**(b)** Set `R0T` to your construct's $R_0^T$ at $P = 3.6$, $D = 0.25$.

**(c)** Set `P_MIN` to the smallest packaging advantage that persists at
$D = 0.25$ in this patient.

**(d)** Your collaborator reports that the burst size in this patient is
**$n = 400$**, twice Table 1's figure. Nothing else about the patient is
remeasured. Compute the new smallest persisting packaging advantage at
$D = 0.25$ **twice**:

- set `P_MIN_N400_T0V0` holding **$T_0$ and $V_0$ fixed** and re-deriving
  $\lambda$ and $k$ from them at the new $n$;
- set `P_MIN_N400_LAMK` holding **$\lambda$ and $k$ fixed** at the values you
  derived for this patient in (a).

Then, in one sentence in the markdown cell below: the two answers to (d) straddle
your answer to (c). Say which of the two is a coherent perturbation of *this*
model and why the other is not. *(Not separately graded.)*

In [ ]:
D_DEATH, T_0, N_BURST, C_CLEAR = 0.02, 800.0, 200.0, 30.0
# Table 1 prints V_0 per MILLILITRE and everything else per microlitre.
V_0 = 8.0e4 / 1000.0        # = 80 virions / ul
P_CONSTRUCT, D_EFFECT = 3.6, 0.25

...

R0_PATIENT, R0T, P_MIN, P_MIN_N400_T0V0, P_MIN_N400_LAMK

In [ ]:
grader.check("q8")

---
## Question 9 — Design the cut, then say what it costs  *(7 points)*

Xie *et al.* (Science **333**, 1307, 2011) classify one cell type out of a panel
of seven with five sensor modules and the expression printed on p.1308:

$$\text{miR-21 AND miR-17-30a AND NOT(miR-141) AND NOT(miR-142(3p)) AND NOT(miR-146a)}$$

**HeLa is the only positive line.** Five of the six negatives are themselves
cancer lines, so a false positive here is the wrong cancer, in a patient.

**The panel.** Fig. 2D's bars, digitized, in rel. u. **These are SENSOR module
outputs, not marker levels** — each series is already oriented so that HeLa reads
high, which means the three NOT terms have been applied once by the biology
before anything was measured. The five column headings name **sensor modules**
and are the dict keys you will type; the quantity each one reports is the level
of the miRNA after its module has acted.

| cell line | `P21` | `P17-30a` | `T141` | `T142-3p` | `T146a` |
|---|---|---|---|---|---|
| DAOY | 0.420 | 0.429 | 0.961 | 0.697 | 0.417 |
| HEK293 | 0.010 | 0.078 | 1.106 | 0.993 | 1.230 |
| HeLa | 0.933 | 0.717 | 0.894 | 0.849 | 0.854 |
| MCF-7 | 0.931 | 0.405 | 0.106 | 1.019 | 1.033 |
| SH-SY5Y | 0.090 | 0.242 | 1.072 | 0.963 | 1.063 |
| SKBR3 | 0.525 | 0.755 | 0.544 | 0.817 | 0.938 |
| T47D | 0.560 | 0.690 | 0.789 | 0.994 | 0.970 |

**The convention this question wants, and it is not optional.** Several bars
exceed 1.0 rel. u., so the table is **normalized first**: divide each column by
its own maximum across these seven lines, which makes $x_m(i) = 1.0$ mean *the
most of marker $m$ seen in* **these** *lines*. Then convert the three NOT'd
sensors back to marker levels with $x = 1 - (\text{sensor output})$ and declare
them `absent`, so that p.1308's three NOT terms are in your code rather than
hidden in the data. Four lines, and type them in this order:

```python
RAW    = {ln: dict(zip(MARKERS, row)) for ln, row in zip(LINES, TABLE)}       # 1
NORM   = cf.normalize_responses(RAW, MARKERS)                                 # 2
MATRIX = [[NORM[ln][m] for m in MARKERS] for ln in LINES]                     # 3
LEVELS = cf.marker_levels_from_sensor_outputs(MATRIX, LINES, MARKERS, SENSE)  # 4
```

Keeping Fig. 2D's sensor outputs and declaring every sense `high` is the same
arithmetic and is not wrong — the two differ by at most $7 \times 10^{-18}$ and
the hidden tests accept either — but type the four lines above, because the rest
of the question is written in marker levels.

**The classifier.** AND is multiplication, not a gate:

$$y(i) \;=\; \prod_{m} \begin{cases} x_m(i) & \text{sense } \texttt{high}\\[2pt]
1 - x_m(i) & \text{sense } \texttt{absent}\end{cases}$$

Every factor lies in $[0, 1]$, so $y(i)$ does too, and nothing has been decided
yet. Write $y_+ = y(\text{HeLa})$.

**The cut.** The deciding is one comparison, made outside the circuit, and it is
yours. Following Xie's own caption — *"input combinations that result in 20%
output relative to HeLa cells"* — the cut is always a **fraction of the positive
line's own output**. Writing that fraction $c$, line $i$ is called positive when

$$y(i) \;\ge\; c\,y_+ ,$$

and a line sitting **exactly on** the cut is called **positive**. That is
`cf.confusion(..., relative=True)`, which is the default. Note this is a
*relative* cut: the comparison moves with $y_+$, so it is not a fixed number of
output units.

**This question fixes the grid of cuts.** You will evaluate the classifier at
exactly four cuts, in this order:

$$c \in \{\,0.25,\; 0.45,\; 0.60,\; 0.72\,\} \quad\text{(that is, 25\%, 45\%, 60\% and 72\% of } y_+\text{)}.$$

Do not search over any others; the answer to this question depends on the grid,
and the grid is given.

**Q9.**

**(a)** Using **all five** sensor modules, set `Y_POS` to $y_+$, and set
`MARGIN` to $y_+$ divided by the largest output among the six negative lines.
Then, in the same cell, confirm with
`cf.convention_check(LEVELS, MARKERS, SENSE, "HeLa").alternative is None` that
your table is in the convention you declared. *(The `alternative` field is the
one that answers that narrow question; do not test whether
`cf.sense_check(...)` is empty, which asks a different and stronger one.)*

**(b)** Still with all five modules, set three 4-tuples, each in the order the
four cuts are printed above:

- `FP_AT_CUTS` — the number of negative **cell lines** called positive;
- `SPEC_AT_CUTS` — the specificity, $\mathrm{TN}/N_-$;
- `SENS_AT_CUTS` — the sensitivity, $\mathrm{TP}/(\mathrm{TP}+\mathrm{FN})$.

$N_- = 6$ is the number of negative lines. **All four counts are counts of cell
lines**, not of cells. There is one positive line, so the sensitivity can only
take two values.

**(c)** One cut on the grid calls no negative line. Set `CUT_ZERO_FP` to that
cut, as the fraction of $y_+$ it is (so one of `0.25`, `0.45`, `0.60`, `0.72`).

Six negative lines, all called correctly, do **not** establish that the per-cell
false-positive rate is zero. Treating the lines as independent draws, the exact
one-sided $100(1-\alpha)\%$ upper limit on a per-cell rate
$\varphi_{\text{cell}}$ when **no** negative line is called is

$$\varphi_{95} \;=\; 1 - \alpha^{1/N_-}, \qquad \alpha = 0.05 .$$

Set `PHI95_ZERO` to that number at $N_- = 6$.

**(d)** Now the clinical arithmetic, and two of its three numbers are **yours**.

$$N_{\text{wrong}} \;=\; \varphi_{\text{cell}} \cdot N_{\text{dose}}$$

against a budget of **one wrongly acted-on cell** — the same product, the same
budget of one, and the opposite sign from the escapers-per-dose arithmetic this
set opened on.

This protocol delivers the circuit to $N_{\text{dose}} = 2.0 \times 10^{8}$
cells. Set `N_DOSE` to the treated population you are multiplying by: that
figure, unless you argue for a different one in the next question. **The hidden
tests check your arithmetic and its direction against whatever you set, never
against a value of ours.**

Set `PHI_CELL` to the per-**cell** false-positive rate you are prepared to defend
at the cut you chose in (c). It is a probability in $[0, 1]$, and **it may not be
zero**: six negative lines resolve no nonzero rate finer than $1/N_- = 1/6$, so a
false-positive count of zero means *below this panel's resolution* and never
*zero*. $\mathrm{FP}/N_-$ is a fraction of cell **lines** and
$\varphi_{\text{cell}}$ is a fraction of **cells** — they are different
quantities, and **nothing in the code can catch you substituting one for the
other**, because both are numbers in $[0, 1]$. `cf.wrong_cells_per_dose` will
multiply whatever you hand it. The substitution is the error the next question
is about; it is yours to avoid.

Then set, with `cf.wrong_cells_per_dose`:

- `N_WRONG` — wrongly acted-on cells at **your** $\varphi_{\text{cell}}$;
- `N_WRONG_BOUND` — wrongly acted-on cells at `PHI95_ZERO`, the rate the clean
  panel's own upper limit permits;
- `MEETS_AT_BOUND` — `True` or `False`: does this design clear a budget of one
  wrongly acted-on cell **at that upper limit**?

Then one line in the markdown cell below, not separately graded: say what
$N_{\text{dose}} = 2.0 \times 10^{8}$ would have to be a claim about for
`N_WRONG` to be a number about a patient. The next question is where that
argument is marked.

In [ ]:
LINES = ["DAOY", "HEK293", "HeLa", "MCF-7", "SH-SY5Y", "SKBR3", "T47D"]
MARKERS = ["P21", "P17-30a", "T141", "T142-3p", "T146a"]
SENSE = {"P21": "high", "P17-30a": "high",
         "T141": "absent", "T142-3p": "absent", "T146a": "absent"}
TABLE = [
    [0.420, 0.429, 0.961, 0.697, 0.417],   # DAOY
    [0.010, 0.078, 1.106, 0.993, 1.230],   # HEK293
    [0.933, 0.717, 0.894, 0.849, 0.854],   # HeLa
    [0.931, 0.405, 0.106, 1.019, 1.033],   # MCF-7
    [0.090, 0.242, 1.072, 0.963, 1.063],   # SH-SY5Y
    [0.525, 0.755, 0.544, 0.817, 0.938],   # SKBR3
    [0.560, 0.690, 0.789, 0.994, 0.970],   # T47D
]
CUTS = (0.25, 0.45, 0.60, 0.72)
N_NEG = 6

...

Y_POS, MARGIN, FP_AT_CUTS, SPEC_AT_CUTS, SENS_AT_CUTS, CUT_ZERO_FP, PHI95_ZERO

In [ ]:
grader.check("q9")

---
## Question 10 — What the panel cannot establish  *(5 points)*

This question is written, not computed. It refers to the previous question's
panel, its four cuts and your own answers. No new arithmetic is required, and no
credit is available for repeating arithmetic you have already done.

Three numbers you may need are printed here so that you are arguing with them
rather than deriving them.

- At $\mathrm{FP} = 0$ out of $N_- = 6$, the exact one-sided 95% upper limit on
  the per-cell rate is the $\varphi_{95}$ you computed in Q9(c).
- For **any** count out of six, the same one-sided limit (the general
  Clopper–Pearson form, which you are given and are not asked to derive) is:

  | $\mathrm{FP}$ out of 6 | 0 | 1 | 2 | 3 |
  |---|---|---|---|---|
  | $\varphi_{95}$ | 0.393 | 0.582 | 0.729 | 0.847 |

  Use the row that matches **your own** count at the lowest cut on your grid.
- An exhaustive search over the five modules and the same four cuts, run against
  a budget of one wrongly acted-on cell, reports a classifier that **meets** the
  budget: two modules at the 60% cut, no false-positive lines, and
  **`wrong_cells = 0.0`**.

<!-- BEGIN QUESTION -->

**Q10a.** Moving the cut from the lowest to the highest on your
grid changes the classifier. Say what it **buys** and what it **costs**: name
the negative lines that leave the false-positive set and the cut at which each
leaves, say what happens to the one positive line and why, and say where the
cost of the highest cut is actually paid, given what your own `SENS_AT_CUTS`
does across the grid. Three or four sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Q10b.** The search result above reports `wrong_cells = 0.0` for a
design that treats $2 \times 10^{8}$ cells. Say why that number may not be
reported, in three parts:

**(i)** what quantity $\mathrm{FP}/N_-$ actually is, what quantity
$\varphi_{\text{cell}}$ has to be for $\varphi_{\text{cell}} \cdot
N_{\text{dose}}$ to mean anything, and what has to be assumed to put one where
the other belongs;

**(ii)** what the finest nonzero rate six negative lines can express is, and
therefore what a reported count of zero false positives is a statement about;

**(iii)** in which direction $\varphi_{95}$ constrains the rate, and what it
therefore licenses you to say — and not say — about a dose of
$2 \times 10^{8}$ cells. Use your own $\varphi_{95}$ from Q9(c) and the 0.729
above. Say also what $N_{\text{dose}}$ itself would have to be a claim about for
the product to be a number about a patient.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Q10c.** Every response in the panel was measured in a cell line
**on its own**. Xie's Fig. 3C scores this circuit's killing efficiency in HEK293
alone at **0.145**; Fig. 4B puts HeLa and HEK293 **in one dish** and the same
quantity in HEK293 comes out at **0.360**. Give the factor between them, and say
what it does to a specificity you computed one line at a time. One or two
sentences.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---
## Question 11 — required for BioE 247, extra credit for BioE 147

**BioE 247** — part of the assignment, worth 8 of your 58
points.

**BioE 147** — optional, worth up to 8 points of extra credit on top of
50. You do not need it for full marks. It is not a harder version of the
same thing; it removes an assumption the earlier questions made, which is
where most of the interest is.

This set has asked you for the same product twice, six questions
apart and with the sign reversed.

In **Question 3** you multiplied a measured escape rate by a dose and compared it
to one, and the thing you were counting was a cell that should have died and did
not. In **Question 9** you multiplied a false-positive rate by a dose and
compared it to one, and the thing you were counting was a cell that should have
lived and did not.

Question 9 made you confront something Question 3 let you get away with: the rate
you multiply by is an **estimate from a finite experiment**, and what the
experiment licenses is a bound, not a point.

<!-- BEGIN QUESTION -->

**Q11.**

**(a)** *(2 points)* Write the two calculations as one expression, and set out
the correspondence term by term: what plays the role of the rate in each, what
plays the role of the population, and what the budget of one means in each. Then
state the one structural difference — not a difference of values, a difference in
what the two numbers are **about**. Four or five sentences.

**(b)** *(3 points)* Question 3 took the plating assay's fraction viable,
$10^{-8.6}$, as a number. Apply Question 9's discipline to it.

**(i)** Say what you would need to know about that plating experiment — name the
two quantities — before you could put a one-sided upper limit on the escape rate
the way $\varphi_{95}$ does for the panel.

**(ii)** Suppose the upper limit turned out to be three-fold above the point
estimate, which is unremarkable for a plating assay. Recompute `ESCAPERS` and
`DOSE_MAX` at that limit and say, in one sentence each, what happens to the
containment claim and to the largest defensible dose.

**(iii)** `RETENTION` was about 1.36 generations. Say whether an upper limit on
the *escape rate* changes it, and why or why not. This one is a trap and the
answer is short.

**(c)** *(3 points)* You are advising on which of the two systems to take into a
first-in-human study, and you may fund **one** additional experiment on **one**
of them.

Name the experiment, name the system, and say what it would have to return to
change a decision. Your answer must turn on which of the two rates is currently
the worse characterized **relative to the budget it has to meet** — so say what
each rate's bound would have to be for its system to clear a budget of one, and
compare that to where each one actually sits. An answer that funds the more
interesting experiment rather than the more decision-relevant one does not get
these marks.

_Type your answer here, replacing this text._

<!-- END QUESTION -->

---

### Before you submit

Restart the kernel and run every cell from the top. The autograder runs your
notebook in a clean environment, and a notebook that only works in the order you
happened to execute it will not pass.

**One closing thought, which is the set's spine.** Every question here asked what
a design does to a population you are not watching.

Question 3 found that a containment claim licensed by a plating assay is a claim
about a single shift in the lab — `RETENTION` was 1.36 generations, which at the
doubling time of Question 2's own control strain is **under an hour**. Question 7 found that the fraction of a
prep that is vector and the fraction of the *infectious* output that is vector
are different numbers about the same tube. Question 8 found that a floor moves
when you perturb a parameter two other parameters were back-solved from, and that
which way it moves depends on what you decided to hold still. Question 9 found
that six clean cell lines and $2 \times 10^{8}$ cells are not the same kind of
evidence, and that the gap between them is most of the distance to a patient.

None of these is a modelling result. Each of them is a number that was already
sitting in the paper, waiting for someone to multiply it by the size of the
population it was going to be applied to.